# 06. 머신러닝 - 이진분류

> 다중분류(06_머신러닝_분류_다중.ipynb)와 모델 코드 구조는 비슷하지만 목적과 옵션이 달라 별도 파일로 분리했습니다. Titanic 생존 예측(생존=1, 사망=0)으로 실습합니다.

### 이 노트북에서 배우는 것
- 이진분류와 다중분류가 무엇이 다른지 먼저 이해하기
- 배깅/부스팅 앙상블 개념과 대표 모델들의 하이퍼파라미터
- 이진분류 모델(Logistic/Tree/RandomForest/GB/XGB/LGBM/SGD/KNN/SVC) 실습
- accuracy/precision/recall/f1/roc_auc 등 이진분류 평가지표
- confusion_matrix, ROC curve로 성능 시각화
- 불균형 데이터에서 재현율이 안 나올 때의 대응법(class_weight, SMOTE)

### 사용 데이터
- `data/train.csv` (Titanic) - Survived(0/1) 예측

## 목차
1. 이진분류란? 다중분류와 무엇이 다른가
2. 이진분류 모델·모듈·평가함수 총정리표
3. 앙상블 개념: 배깅 vs 부스팅
4. 이진분류 모델 실습
5. 이진분류 평가지표
6. confusion_matrix / ROC curve 시각화
7. Feature Importance 해석
8. 여러 모델 앙상블 (VotingClassifier)
9. 여러 모델 성능 비교 시각화
10. 불균형 데이터 & 재현율 개선
11. 종합 실전 문제

In [ ]:
import lightgbm  # Windows에서 numpy/pandas보다 나중에 불러오면 DLL 충돌이 날 수 있어 가장 먼저 import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../data/train.csv')
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Sex'] = df['Sex'].map({'male': 1, 'female': 0})
df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)

feature_cols = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare'] + [c for c in df.columns if c.startswith('Embarked_')]
X = df[feature_cols]
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(X_train.shape, X_test.shape, y_train.mean().round(3), y_test.mean().round(3))

## 1. 이진분류란? 다중분류와 무엇이 다른가

### 📖 이론 설명
**이진분류**는 클래스가 정확히 2개(예: 생존/사망, 이탈/유지)일 때, **다중분류**는 클래스가 3개 이상(예: 품종 A/B/C)일 때를 말합니다. 코드 구조는 거의 같지만 두 가지가 달라집니다.

1. **출력 형태**: 이진분류는 '양성 클래스일 확률' 하나만 있으면 되지만(`predict_proba(X)[:, 1]`), 다중분류는 클래스별 확률 벡터가 필요합니다.
2. **평가지표 옵션**: `precision_score`, `recall_score`, `f1_score`는 이진분류에서는 기본값으로 바로 계산되지만, 다중분류에서는 클래스별 점수를 하나로 합칠 방법(`average='macro'`/`'weighted'`/`'micro'`)을 반드시 지정해야 합니다.

이 노트북(06)은 **이진분류**, 07번 노트북은 **다중분류**를 다루며, 07번에서 이 차이를 다시 코드로 확인합니다.

## 2. 이진분류 모델·모듈·평가함수 총정리표

### 🧩 핵심 개념 정리
| 모델명 | 모듈 |
|---|---|
| LogisticRegression | sklearn.linear_model |
| DecisionTreeClassifier | sklearn.tree |
| RandomForestClassifier | sklearn.ensemble |
| GradientBoostingRegressor(분류는 GradientBoostingClassifier) | sklearn.ensemble |
| XGBClassifier | xgboost |
| LGBMClassifier | lightgbm |
| SGDClassifier | sklearn.linear_model |
| KNeighborsClassifier | sklearn.neighbors |
| SVC | sklearn.svm |

평가함수: `accuracy_score`, `precision_score`, `recall_score`, `f1_score`, `roc_auc_score`, `confusion_matrix`, `classification_report` (모두 `sklearn.metrics`)

## 3. 앙상블 개념: 배깅 vs 부스팅

### 📖 이론 설명
**배깅(Bagging)**은 원본 데이터에서 중복을 허용해 여러 번 샘플링한 뒤, 각각 독립적으로 모델을 학습시키고 **다수결/평균**으로 최종 예측을 냅니다. 대표적으로 **RandomForest**가 있습니다 - 여러 개의 DecisionTree를 서로 다른 데이터·특성 조합으로 학습시켜 과적합을 줄입니다.

**부스팅(Boosting)**은 모델을 순차적으로 학습시키며, **이전 모델이 잘못 맞춘 데이터에 가중치를 더 주어** 다음 모델이 그 부분을 더 신경 쓰게 만듭니다. **XGBoost, LGBM, GradientBoosting**이 대표적입니다. 배깅보다 성능이 좋은 경우가 많지만 학습 순서가 있어 병렬화가 어렵고 과적합에 더 민감합니다.

### 🧩 핵심 개념 정리 - 주요 하이퍼파라미터
| 파라미터 | 의미 | 해당 모델 |
|---|---|---|
| n_estimators | 트리(모델)의 개수 | RandomForest, XGBoost, LGBM |
| max_depth | 트리 최대 깊이(과적합 방지) | 전체 트리 계열 |
| learning_rate | 학습률(부스팅 단계별 반영 비율) | XGBoost, LGBM, GradientBoosting |
| random_state | 랜덤 시드 고정 | 전체 |
| n_jobs | 병렬 처리에 사용할 CPU 개수 | RandomForest, XGBoost 등 |

## 4. 이진분류 모델 실습

### 📖 이론 설명
모델링 5단계 템플릿을 그대로 적용해 여러 분류 모델을 학습시켜 비교합니다.

### 💻 예제 코드

In [ ]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

models = {
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'DecisionTree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss'),
    'LGBM': LGBMClassifier(n_estimators=100, random_state=42, verbose=-1),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVC': SVC(probability=True, random_state=42),
}

preds = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds[name] = model.predict(X_test_scaled)
print('학습 완료:', list(models.keys()))

### ✍️ TODO: 실전 문제

**문제 1.** `SGDClassifier(penalty='l1', random_state=42)`를 학습시키고 예측값 `sgd_pred`를 만드세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
sgd = SGDClassifier(penalty='l1', random_state=42)  # penalty='l1': 불필요한 가중치를 0으로 만들어 변수 선택 효과를 주는 규제
sgd.fit(X_train_scaled, y_train)  # SGD 계열은 스케일링이 안 된 데이터에서 학습이 불안정하므로 스케일링된 데이터를 사용
sgd_pred = sgd.predict(X_test_scaled)
```

</details>

## 5. 이진분류 평가지표

### 📖 이론 설명
- **accuracy**: 전체 중 맞춘 비율. 클래스가 불균형하면 오해를 줄 수 있음(9절에서 다룸).
- **precision(정밀도)**: '양성이라고 예측한 것 중' 실제 양성 비율. 오탐(false positive)을 줄이는 게 중요할 때.
- **recall(재현율)**: '실제 양성 중' 맞춘 비율. 놓치면 안 되는 상황(질병 진단, 이탈 예측)에서 중요.
- **f1-score**: precision과 recall의 조화평균. 둘 사이 균형을 볼 때.
- **roc_auc**: 임계값을 바꿔가며 본 전반적인 분류 성능(1에 가까울수록 좋음).

### 🧩 핵심 개념 정리
| 함수 | 무엇을 재는가 |
|---|---|
| accuracy_score | 전체 정확도 |
| precision_score | 양성 예측의 정확도 |
| recall_score | 실제 양성을 놓치지 않는 정도 |
| f1_score | precision·recall 조화평균 |
| roc_auc_score | 임계값 무관 전반적 분류력 |

### 💻 예제 코드

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

rf_model = models['RandomForest']
rf_pred = preds['RandomForest']
rf_proba = rf_model.predict_proba(X_test_scaled)[:, 1]

print('accuracy :', accuracy_score(y_test, rf_pred))
print('precision:', precision_score(y_test, rf_pred))
print('recall   :', recall_score(y_test, rf_pred))
print('f1       :', f1_score(y_test, rf_pred))
print('roc_auc  :', roc_auc_score(y_test, rf_proba))
print('model.score():', rf_model.score(X_test_scaled, y_test))  # accuracy와 동일

### ✍️ TODO: 실전 문제

**문제 1.** `XGBoost` 모델의 accuracy, f1-score, roc_auc를 구하세요. (`predict_proba` 필요)

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
xgb_model = models['XGBoost']
xgb_pred = preds['XGBoost']
xgb_proba = xgb_model.predict_proba(X_test_scaled)[:, 1]  # roc_auc는 0/1 예측이 아니라 '확률'이 필요하므로 predict_proba의 양성 클래스 확률(열 1)만 꺼냄
print(accuracy_score(y_test, xgb_pred))
print(f1_score(y_test, xgb_pred))
print(roc_auc_score(y_test, xgb_proba))
```

</details>

## 6. confusion_matrix / ROC curve 시각화

### 📖 이론 설명
혼동행렬(confusion matrix)은 [실제 0 예측 0], [실제 0 예측 1], [실제 1 예측 0], [실제 1 예측 1] 네 가지 경우의 개수를 표로 보여줍니다. `sns.heatmap`으로 그릴 수도 있고, sklearn이 제공하는 `ConfusionMatrixDisplay`를 쓰면 축 이름을 자동으로 붙여줍니다. ROC curve는 임계값을 0~1로 바꿔가며 '진짜 양성률(TPR)'과 '가짜 양성률(FPR)'이 어떻게 변하는지 그린 곡선으로, 곡선 아래 면적(AUC)이 넓을수록 좋은 모델입니다.

### 🧩 핵심 개념 정리
| 함수 | 설명 |
|---|---|
| confusion_matrix(y_true, y_pred) | 2x2(이진) 혼동행렬 |
| ConfusionMatrixDisplay | sklearn 내장 시각화 |
| roc_curve(y_true, y_score) | (fpr, tpr, thresholds) 반환 |
| classification_report | precision/recall/f1 한 번에 |

### 💻 예제 코드

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, classification_report

cm = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds')
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('RandomForest Confusion Matrix')
plt.show()

ConfusionMatrixDisplay.from_predictions(y_test, rf_pred)
plt.show()

fpr, tpr, _ = roc_curve(y_test, rf_proba)
plt.plot(fpr, tpr, label=f'RandomForest (AUC={roc_auc_score(y_test, rf_proba):.3f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title('ROC Curve')
plt.legend(); plt.show()

print(classification_report(y_test, rf_pred))

### ✍️ TODO: 실전 문제

**문제 1.** `LogisticRegression` 모델의 confusion matrix를 heatmap으로 그리세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
logi = models['LogisticRegression']
logi_pred = preds['LogisticRegression']
cm2 = confusion_matrix(y_test, logi_pred)
sns.heatmap(cm2, annot=True, fmt='d', cmap='Blues')  # fmt='d': 칸 안의 숫자를 정수로 표시(기본값은 소수점이 붙어 지저분함)
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.show()
```

</details>

## 7. Feature Importance 해석

### 📖 이론 설명
트리 계열 모델의 `feature_importances_`로 어떤 변수가 생존 예측에 가장 크게 기여했는지 확인합니다.

### 💻 예제 코드

In [ ]:
rf_imp = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(rf_imp)
plt.barh(rf_imp.index, rf_imp.values)
plt.title('Feature Importance (RandomForest)')
plt.show()

## 8. 여러 모델 앙상블 (VotingClassifier)

### 📖 이론 설명
서로 다른 모델들의 예측을 종합하면, 개별 모델의 약점이 상쇄되어 더 안정적인 예측을 얻을 수 있는 경우가 많습니다. `VotingClassifier`는 `(이름, 모델)` 튜플의 리스트를 받아 여러 모델을 하나처럼 다룹니다. `voting='hard'`는 각 모델의 예측 클래스 중 다수결로 정하고, `voting='soft'`는 각 모델이 예측한 확률(`predict_proba`)의 평균으로 최종 클래스를 정합니다 (soft를 쓰려면 리스트에 포함된 모든 모델이 `predict_proba`를 지원해야 하므로, 확률 출력을 지원하지 않는 `SVC`는 `probability=True` 옵션을 켜줘야 합니다).

### 💻 예제 코드

In [ ]:
from sklearn.ensemble import VotingClassifier

voting_models = [('logi', LogisticRegression(max_iter=1000)), ('rf', RandomForestClassifier(n_estimators=100, random_state=42))]
voting = VotingClassifier(voting_models, voting='soft')
voting.fit(X_train_scaled, y_train)
voting_pred = voting.predict(X_test_scaled)
print('Voting Accuracy:', accuracy_score(y_test, voting_pred))

### ✍️ TODO: 실전 문제

**문제 1.** `LogisticRegression`, `RandomForest`, `XGBoost` 세 모델을 `voting='soft'`인 `VotingClassifier`로 묶어 학습시키고 accuracy를 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
voting2 = VotingClassifier(
    [('logi', LogisticRegression(max_iter=1000)),
     ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
     ('xgb', XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss'))],
    voting='soft'
)  # voting='soft': 각 모델의 예측 확률을 평균 내서 최종 클래스를 정함(hard는 단순 다수결)
voting2.fit(X_train_scaled, y_train)
voting2_pred = voting2.predict(X_test_scaled)
print(accuracy_score(y_test, voting2_pred))
```

</details>

## 9. 여러 모델 성능 비교 시각화

### 📖 이론 설명
모델을 하나씩 따로 보는 것보다, 여러 모델의 지표를 한 번에 막대그래프로 나란히 비교하면 어떤 모델이 어떤 지표에서 강점이 있는지 빠르게 파악할 수 있습니다.

### 💻 예제 코드

In [ ]:
score_table = pd.DataFrame({
    name: {
        'accuracy': accuracy_score(y_test, pred),
        'f1': f1_score(y_test, pred),
        'recall': recall_score(y_test, pred),
    } for name, pred in preds.items()
}).T.sort_values('f1', ascending=False)
print(score_table)

score_table['f1'].plot(kind='barh')
plt.title('Model Comparison - F1 Score')
plt.show()

## 10. 불균형 데이터 & 재현율 개선

### 📖 이론 설명
Titanic 데이터는 생존 38% : 사망 62% 정도로 심하게 불균형하지는 않지만, 실무 데이터(이탈 예측, 사기 탐지 등)는 소수 클래스가 5% 미만인 경우도 흔합니다. 이럴 때 모델이 **'전부 다수 클래스로 찍어도' accuracy가 높게 나오는** 함정에 빠지기 쉽습니다 - 그래서 불균형 데이터에서는 accuracy보다 recall/f1/roc_auc를 봐야 합니다.

대응법은 두 가지입니다.
1. **class_weight='balanced'**: 소수 클래스의 오분류에 더 큰 패널티를 줘서 모델이 소수 클래스를 더 신경 쓰게 만듦.
2. **SMOTE(오버샘플링)**: 소수 클래스 데이터를 '주변 데이터를 보고 새로 합성'해서 늘려, 학습 데이터의 클래스 비율을 맞춤.

재현율을 높이면 보통 정밀도(precision)가 떨어지는 **트레이드오프**가 발생하므로, 어느 지표가 더 중요한 문제인지 먼저 판단해야 합니다. (더 극단적으로 불균형한 데이터는 `notebooks_extra/EX03`에서 고객이탈 데이터로 깊게 다룹니다.)

### 🧩 핵심 개념 정리
| 방법 | 코드 |
|---|---|
| class_weight | `RandomForestClassifier(class_weight='balanced')` |
| SMOTE | `from imblearn.over_sampling import SMOTE` |

### 💻 예제 코드

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

# 1) class_weight='balanced'
rf_balanced = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_balanced.fit(X_train_scaled, y_train)
pred_balanced = rf_balanced.predict(X_test_scaled)
print('class_weight 적용 recall:', recall_score(y_test, pred_balanced))

# 2) SMOTE로 오버샘플링
print('SMOTE 적용 전:', y_train.value_counts().to_dict())
smote = SMOTE(random_state=42)
X_train_over, y_train_over = smote.fit_resample(X_train_scaled, y_train)
print('SMOTE 적용 후:', pd.Series(y_train_over).value_counts().to_dict())

rf_smote = RandomForestClassifier(n_estimators=100, random_state=42)
rf_smote.fit(X_train_over, y_train_over)
pred_smote = rf_smote.predict(X_test_scaled)
print('SMOTE 적용 recall:', recall_score(y_test, pred_smote))
print('SMOTE 적용 precision:', precision_score(y_test, pred_smote))  # recall이 오르면 precision은 대체로 떨어짐

### ✍️ TODO: 실전 문제

**문제 1.** `LogisticRegression(class_weight='balanced', max_iter=1000)`을 학습시켜 recall을 기본 LogisticRegression과 비교하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
logi_balanced = LogisticRegression(class_weight='balanced', max_iter=1000)  # class_weight='balanced': 소수 클래스의 오분류에 더 큰 패널티를 줘서 재현율을 끌어올림
logi_balanced.fit(X_train_scaled, y_train)
pred_lb = logi_balanced.predict(X_test_scaled)
print('balanced recall:', recall_score(y_test, pred_lb))
print('기본 recall:', recall_score(y_test, preds['LogisticRegression']))  # 두 recall을 나란히 비교해 class_weight 효과를 직접 확인
```

</details>

## 11. 종합 실전 문제

**이진분류 전체 흐름을 이어서 풀어보는 미니 모의고사입니다.**

**문제 1.** `XGBClassifier(n_estimators=150, max_depth=4, random_state=42)`를 학습시키고 accuracy, f1-score를 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
xgb2 = XGBClassifier(n_estimators=150, max_depth=4, random_state=42, eval_metric='logloss')  # eval_metric을 직접 지정하지 않으면 최신 버전에서 경고가 뜨는 경우가 있어 명시
xgb2.fit(X_train_scaled, y_train)
pred2 = xgb2.predict(X_test_scaled)
print(accuracy_score(y_test, pred2))
print(f1_score(y_test, pred2))
```

</details>

**문제 2.** `RandomForestClassifier`의 confusion matrix를 구하고, False Negative(실제 생존인데 사망으로 예측한) 개수를 `fn_count`에 저장하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
cm3 = confusion_matrix(y_test, rf_pred)  # confusion_matrix는 [[TN, FP], [FN, TP]] 순서로 반환됨
fn_count = cm3[1, 0]  # [1,0] 위치가 바로 False Negative(실제 1인데 0으로 예측)
print(fn_count)
```

</details>